# Data Cleaning & Feature Engineering

This notebook performs the cleaning, standardization, and feature engineering of the LendingClub dataset.

### Phase 1: Setup & Data Loading

In [8]:
import pandas as pd
import numpy as np
import os

# Define raw data path
raw_data_path = '../data/raw/lending_club_loan_two.csv'

# Fallback for different execution environments (root vs notebooks folder)
if not os.path.exists(raw_data_path):
    raw_data_path = 'data/raw/lending_club_loan_two.csv'

df = pd.read_csv(raw_data_path)
df_copy = df.copy()
print(f"Original dataset shape: {df.shape}")

Original dataset shape: (396030, 27)


### Phase 2: Schema Standardization

In [10]:
# 2.1 Column Renaming
df.rename(columns={
    'loan_amnt': 'loan_amount',
    'int_rate': 'interest_rate',
    'installment': 'monthly_installment',
    'annual_inc': 'annual_income',
    'dti': 'debt_to_income_ratio',
    'open_acc': 'number_of_open_accounts',
    'pub_rec': 'public_records_derogatory',
    'revol_bal': 'revolving_balance',
    'revol_util': 'revolving_line_utilization',
    'total_acc': 'total_credit_lines',
    'pub_rec_bankruptcies': 'public_record_bankruptcies',
    'issue_d': 'issue_date',
    'earliest_cr_line': 'earliest_credit_line_opened',
    'term': 'loan_term',
    'grade': 'loan_grade',
    'sub_grade': 'loan_sub_grade',
    'home_ownership': 'home_ownership_status',
    'verification_status': 'income_verification_status',
    'loan_status': 'loan_repayment_status',
    'purpose': 'loan_purpose',
    'title': 'loan_title_description',
    'zip_code': 'zip_code_prefix',
    'addr_state': 'borrower_state_address',
    'initial_list_status': 'initial_listing_status',
    'application_type': 'loan_application_type',
    'address': 'borrower_full_address',
    'log_income': 'log_annual_income',
    'mort_acc':'number_of_mortgage_accounts'
}, inplace=True)

print("Columns after renaming:")
print(df.columns)

Columns after renaming:
Index(['loan_amount', 'loan_term', 'interest_rate', 'monthly_installment',
       'loan_grade', 'loan_sub_grade', 'emp_title', 'emp_length',
       'home_ownership_status', 'annual_income', 'income_verification_status',
       'issue_date', 'loan_repayment_status', 'loan_purpose',
       'loan_title_description', 'debt_to_income_ratio',
       'earliest_credit_line_opened', 'number_of_open_accounts',
       'public_records_derogatory', 'revolving_balance',
       'revolving_line_utilization', 'total_credit_lines',
       'initial_listing_status', 'loan_application_type',
       'number_of_mortgage_accounts', 'public_record_bankruptcies',
       'borrower_full_address'],
      dtype='str')


### Phase 3: Structural Cleaning

In [11]:
# 3.1 Drop Redundant Columns
df.drop(['emp_title', 'emp_length', 'number_of_mortgage_accounts'], axis=1, inplace=True)

# 3.2 Handle Duplicates
df.drop_duplicates(inplace=True)
print(f"Dataset shape after structural cleaning: {df.shape}")

Dataset shape after structural cleaning: (396030, 24)


### Phase 4: Missing Value Treatment

In [12]:
df['revolving_line_utilization'].fillna(df['revolving_line_utilization'].median(), inplace=True)
df['public_record_bankruptcies'].fillna(df['public_record_bankruptcies'].median(), inplace=True)
df['loan_title_description'].fillna("Unknown", inplace=True)
df.dropna(inplace=True)

print("Missing values handled. Null count:")
print(df.isnull().sum().sum())

Missing values handled. Null count:
0


/var/folders/5w/q221yrhd6c99fl93ycz7_nk80000gn/T/ipykernel_11472/2679402923.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['revolving_line_utilization'].fillna(df['revolving_line_utilization'].median(), inplace=True)
/var/folders/5w/q221yrhd6c99fl93ycz7_nk80000gn/T/ipykernel_11472/2679402923.py:2: ChainedAssignmentError: A va

### Phase 5: Data Type & Outlier Cleaning

In [13]:
df['loan_amount'] = df['loan_amount'].astype(float)
df['annual_income'] = df['annual_income'].astype(float)
df['issue_date'] = pd.to_datetime(df['issue_date'], format='%b-%Y')
df['loan_term'] = df['loan_term'].str.strip()

# Outlier Filtering (Annual Income < 99th percentile)
df = df[df['annual_income'] < df['annual_income'].quantile(0.99)]

### Phase 6: Feature Engineering

In [14]:
import numpy as np
import pandas as pd

# ------------------------------------------------
# 1. loan_income_ratio
# Meaning: Loan amount compared to the borrower's yearly income.
# Purpose: Shows how large the loan is relative to income.
# Higher value means the borrower is taking a bigger loan burden.
# ------------------------------------------------
df["loan_income_ratio"] = df["loan_amount"] / df["annual_income"]


# ------------------------------------------------
# 2. installment_income_ratio
# Meaning: Monthly installment compared to monthly income.
# Purpose: Measures how much of the borrower's monthly income
# goes toward loan repayment.
# Higher value means higher monthly repayment pressure.
# ------------------------------------------------
df["installment_income_ratio"] = (
    df["monthly_installment"] / (df["annual_income"] / 12)
)


# ------------------------------------------------
# 3. risk_score
# Meaning: Combined risk indicator using DTI, utilization,
# loan-income ratio, and bankruptcy records.
# Purpose: Gives a single numeric score to compare borrower risk.
# Higher score means the borrower may be financially riskier.
# ------------------------------------------------
df["risk_score"] = (
    0.4 * df["debt_to_income_ratio"] +
    0.3 * df["revolving_line_utilization"] +
    0.2 * df["loan_income_ratio"] * 100 +
    0.1 * df["public_record_bankruptcies"].fillna(0) * 10
)


# ------------------------------------------------
# 4. credit_stress_score
# Meaning: Measures credit pressure using DTI, credit utilization,
# and derogatory public records.
# Purpose: Helps identify borrowers under financial stress.
# Higher score means higher credit stress.
# ------------------------------------------------
df["credit_stress_score"] = (
    df["debt_to_income_ratio"] +
    df["revolving_line_utilization"] +
    (df["public_records_derogatory"].fillna(0) * 10)
)


# ------------------------------------------------
# 5. issue_year
# Meaning: Year in which the loan was issued.
# Purpose: Useful for year-wise trend analysis such as
# loan volume, average loan amount, interest rate, or risk over time.
# ------------------------------------------------
df["issue_date"] = pd.to_datetime(
    df["issue_date"].astype(str),
    format="%b-%Y",
    errors="coerce"
)

df["issue_year"] = df["issue_date"].dt.year


# ------------------------------------------------
# 6. income_band
# Meaning: Groups borrowers into income categories.
# Purpose: Makes income-based EDA easier by comparing
# low, middle, and high income borrowers.
# ------------------------------------------------
df["income_band"] = pd.cut(
    df["annual_income"],
    bins=[0, 40000, 100000, np.inf],
    labels=["Low", "Middle", "High"]
)


# ------------------------------------------------
# 7. dti_band
# Meaning: Groups borrowers based on debt-to-income ratio.
# Purpose: Helps compare borrowers with low, medium,
# and high debt burden.
# ------------------------------------------------
df["dti_band"] = pd.cut(
    df["debt_to_income_ratio"],
    bins=[0, 10, 20, np.inf],
    labels=["Low", "Medium", "High"]
)


# ------------------------------------------------
# 8. utilization_band
# Meaning: Groups borrowers based on revolving credit utilization.
# Purpose: Helps compare borrowers with low, medium,
# and high credit usage.
# Higher utilization usually means higher financial pressure.
# ------------------------------------------------
df["utilization_band"] = pd.cut(
    df["revolving_line_utilization"],
    bins=[0, 30, 70, 100],
    labels=["Low", "Medium", "High"]
)


df.head()

,loan_amount,loan_term,interest_rate,monthly_installment,loan_grade,loan_sub_grade,home_ownership_status,annual_income,income_verification_status,issue_date,...,public_record_bankruptcies,borrower_full_address,loan_income_ratio,installment_income_ratio,risk_score,credit_stress_score,issue_year,income_band,dti_band,utilization_band
0,10000.0,36 months,11.44,329.48,B,B4,RENT,117000.0,Not Verified,NaT,...,0.0,"0174 Michelle Gateway\r\nMendozaberg, OK 22690",0.085470,0.033793,24.745402,68.04,NaN,High,High,Medium
1,8000.0,36 months,11.99,265.68,B,B5,MORTGAGE,65000.0,Not Verified,NaT,...,0.0,"1076 Carney Fort Apt. 347\r\nLoganmouth, SD 05113",0.123077,0.049049,27.271538,75.35,NaN,Middle,High,Medium
2,15600.0,36 months,10.49,506.97,B,B3,RENT,43057.0,Source Verified,NaT,...,0.0,"87025 Mark Dale Apt. 269\r\nNew Sabrina, WV 05113",0.362310,0.141293,40.022209,104.99,NaN,Middle,Medium,High
3,7200.0,36 months,6.49,220.65,A,A2,RENT,54000.0,Not Verified,NaT,...,0.0,"823 Reid Ford\r\nDelacruzside, MA 00813",0.133333,0.049033,10.156667,24.10,NaN,Middle,Low,Low
4,24375.0,60 months,17.27,609.33,C,C5,MORTGAGE,55000.0,Verified,NaT,...,0.0,"679 Luna Roads\r\nGreggshire, VA 11650",0.443182,0.132945,43.383636,103.75,NaN,Middle,High,Medium


### Phase 7: Quality Audit & Export

In [15]:
# Final Statistics
print("--- Final Dataset Audit ---")
print(f"Final shape: {df.shape}")
display(df.head())

# Define output path
output_path = '../data/processed/cleaned_club_loan_two.csv'
if not os.path.exists('../data'): # Handle root execution
    output_path = 'data/processed/cleaned_club_loan_two.csv'

# 1. Create the directory if it doesn't exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# 2. Export the data
df.to_csv(output_path, index=False)
print(f"\nSuccess! Cleaned dataset with features saved to: {output_path}")

--- Final Dataset Audit ---
Final shape: (389412, 32)


,loan_amount,loan_term,interest_rate,monthly_installment,loan_grade,loan_sub_grade,home_ownership_status,annual_income,income_verification_status,issue_date,...,public_record_bankruptcies,borrower_full_address,loan_income_ratio,installment_income_ratio,risk_score,credit_stress_score,issue_year,income_band,dti_band,utilization_band
0,10000.0,36 months,11.44,329.48,B,B4,RENT,117000.0,Not Verified,NaT,...,0.0,"0174 Michelle Gateway\r\nMendozaberg, OK 22690",0.085470,0.033793,24.745402,68.04,NaN,High,High,Medium
1,8000.0,36 months,11.99,265.68,B,B5,MORTGAGE,65000.0,Not Verified,NaT,...,0.0,"1076 Carney Fort Apt. 347\r\nLoganmouth, SD 05113",0.123077,0.049049,27.271538,75.35,NaN,Middle,High,Medium
2,15600.0,36 months,10.49,506.97,B,B3,RENT,43057.0,Source Verified,NaT,...,0.0,"87025 Mark Dale Apt. 269\r\nNew Sabrina, WV 05113",0.362310,0.141293,40.022209,104.99,NaN,Middle,Medium,High
3,7200.0,36 months,6.49,220.65,A,A2,RENT,54000.0,Not Verified,NaT,...,0.0,"823 Reid Ford\r\nDelacruzside, MA 00813",0.133333,0.049033,10.156667,24.10,NaN,Middle,Low,Low
4,24375.0,60 months,17.27,609.33,C,C5,MORTGAGE,55000.0,Verified,NaT,...,0.0,"679 Luna Roads\r\nGreggshire, VA 11650",0.443182,0.132945,43.383636,103.75,NaN,Middle,High,Medium



Success! Cleaned dataset with features saved to: ../data/processed/cleaned_club_loan_two.csv
